# Thành viên 2 — Kaggle `GPU T4 x 2` (32 GB VRAM): Track Quốc tế `PHEME` (`Twitter-RoBERTa + mDeBERTa Stance Verbalizer -> SBi-FAGAT`)

**Cấu hình bắt buộc trên Kaggle trước khi chạy song song với Thành viên 1:**
1. Bật **Accelerator: GPU T4 x 2** (tổng 32 GB VRAM).
2. Bật **Internet: ON** để tải `PHEME` từ Figshare (có kiểm tra MD5) và trọng số Hugging Face.

Bạn có 2 chế độ chạy song song:
- **`member2_en`** *(Khuyên dùng để chia 50/50 với Thành viên 1 — Tier 2A PHEME `K=10, min_total_replies=15`)*: Chạy trọn gói dữ liệu chuẩn quốc tế `PHEME` (`cardiffnlp/twitter-roberta-base-offensive` + `mDeBERTa-v3-xnli` Stance Verbalizer -> chưng cất Cross-Encoder 2D `cardiffnlp/twitter-roberta-base` -> trích xuất `.pt` *Ancestor SpanPool* -> 5-seed công bằng Inductive + Leave-One-Event-Out + Ablation ở `K=10, N>=15` để mở rộng số cây & số fold LOO hợp lệ; kiểm định stance EN qua RumourEval SDQC).
- **`model_gnn_only`**: Chuyên trách train & tinh chỉnh **Mạng Đồ thị `SBi-FAGAT` + 7 mô hình so sánh** (nếu đã nạp `processed/vn` từ Thành viên 1 hoặc bật `--bootstrap-if-missing` để chạy ngay không cần chờ).

In [ ]:
# Bước 1: Kiểm tra GPU NVIDIA Tesla T4
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available(), "| Num GPUs:", torch.cuda.device_count())

In [ ]:
%%bash
set -euo pipefail
# Bước 2: Clone/cập nhật repo từ GitHub & cài gói (khóa đúng torch CUDA của Kaggle)
BRANCH="main"
cd /kaggle/working
if [ ! -d "gnn-nlp-propagation/.git" ]; then
  rm -rf gnn-nlp-propagation
  git clone https://github.com/hieu-kiien/gnn-nlp-propagation.git
fi
cd /kaggle/working/gnn-nlp-propagation
git fetch origin
git checkout "$BRANCH"
git pull --ff-only origin "$BRANCH"
python -c 'import torch; print("torch==" + torch.__version__.split("+")[0])' > /tmp/kaggle_constraints.txt
grep -v '^torch>=' requirements.txt > /tmp/req_kaggle.txt
pip install -q -c /tmp/kaggle_constraints.txt -r /tmp/req_kaggle.txt
pytest -q

In [ ]:
# Khởi tạo muối băm SHA-256 bền vững (nếu chưa đặt SBIF_HASH_SALT) cho giả danh hóa tác giả PHEME
import os, pathlib, secrets
os.chdir("/kaggle/working/gnn-nlp-propagation")
salt_file = pathlib.Path("data/interim/.author_hash_salt.secret")
if not os.environ.get("SBIF_HASH_SALT") and not salt_file.exists():
    salt_file.parent.mkdir(parents=True, exist_ok=True)
    salt_file.write_text(secrets.token_hex(16), encoding="utf-8")
    print("Created persistent salt at", salt_file)

In [ ]:
# Bước 3: Kiểm thử khói toàn bộ đường ống (xong trong <10 giây)
!python -m scripts.kaggle_parallel_runner --role member2_en --smoke-test

In [ ]:
# Bước 4: CHẠY CHÍNH THỨC SONG SONG TRÊN KAGGLE
# Đổi ROLE = "model_gnn_only" và LANG = "vi" (kèm --processed-dir) nếu chạy nhánh GNN từ dữ liệu Bạn 1
import shlex, subprocess, sys
ROLE = "member2_en"  # hoặc "model_gnn_only"
LANG = "en"          # đổi thành "vi" nếu nạp processed/vn từ Thành viên 1
K_HORIZON = 10 if LANG == "en" else 20
MIN_REPLIES = 15 if LANG == "en" else 25

cmd = (
    f"-m scripts.kaggle_parallel_runner --role {ROLE} --lang {LANG} "
    f"--k-horizon {K_HORIZON} --min-total-replies {MIN_REPLIES} "
    f"--hidden-dim 64 --epochs-nlp 3 --epochs-gnn 100 --run-upfd"
)
subprocess.run([sys.executable, *shlex.split(cmd)], check=True)

In [ ]:
# Bước 5 (Tùy chọn): Chạy bổ sung UPFD PolitiFact (không làm fail Kaggle Commit nếu Google Drive chặn IP)
import pathlib, subprocess, sys, zipfile
res = subprocess.run([
    sys.executable, "-m", "scripts.run_upfd_extension",
    "--dataset-name", "politifact",
    "--feature", "bert",
    "--hidden-dim", "64",
    "--output-table", "paper/tables_upfd_politifact.tex",
    "--output-json", "paper/results_upfd_politifact.json",
])
if res.returncode == 0:
    bundle_zip = pathlib.Path(f"sbifagat_{ROLE}_bundle.zip")
    if bundle_zip.is_file():
        with zipfile.ZipFile(bundle_zip, "a", compression=zipfile.ZIP_DEFLATED) as zf:
            for rel in ("paper/tables_upfd_politifact.tex", "paper/results_upfd_politifact.json"):
                fp = pathlib.Path(rel)
                if fp.is_file():
                    zf.write(fp, arcname=rel)
        print("Appended UPFD PolitiFact artifacts into", bundle_zip)
else:
    print("[WARN] Optional UPFD PolitiFact exited with code", res.returncode, "(primary PHEME bundle preserved).")

## Xuất kết quả gộp với Thành viên 1
Trước khi tắt máy ảo Kaggle, tải **2 gói nén** trong `/kaggle/working/gnn-nlp-propagation/`:
1. `sbifagat_member2_en_bundle.zip` — gồm `paper/results_main_pheme.json`, `paper/results_loo_pheme.json`, `paper/results_ablation_pheme.json`, `paper/results_upfd.json`, `paper/tables_*_pheme.tex`, `paper/tables_upfd.tex`, `checkpoints/best_sbifagat_pheme.pt`, `paper/figures/pheme/*.pdf` và phiếu kiểm định EN (tùy chọn; kiểm định stance tiếng Anh chính thức dùng RumourEval SDQC).
2. `sbifagat_member2_en_sealed_machine_keys.zip` — gói niêm phong riêng chứa `*_machine_key.csv` (giữ riêng để đối chiếu `machine_vs_gold` nếu chạy audit cặp EN).

*Lưu ý:* Ở chế độ mặc định công bằng không chạy `auto_loop` (thêm `--run-auto-loop` ở Bước 4 nếu muốn xuất thêm `outputs/auto_loop_pheme/{loop_diagnostics,worst_val_cases_for_review}.json`).